# Penalties as a game of mixed strategies

A penalty kick is the cleanest natural experiment in football: a one-shot, two-player,
near-simultaneous game. The shooter picks a side, the keeper picks a side, and a goal or
a save falls out of whether they match. Game theory has used it for decades to test
whether real professionals play mixed-strategy equilibria.

The earlier version of this piece had to **apologise** for its data: StatsBomb open data
only records which way a keeper dived when the dive ended in a save, so the keeper's choice
was unobservable on ~3/4 of kicks. This rebuild uses a World Cup-specific corpus that
records the keeper's dive on **every** kick — so we can finally play out the whole game.

Source: every men's World Cup shootout kick 1982→2022 (`data/raw/WorldCupShootouts.csv`).
Both the shot `Zone` (1–9 grid) and the `Keeper` dive (L/C/R) are coded from the shooter's
view of the goal, so "did the keeper read the side" is a direct comparison.

In [1]:
import pandas as pd

from mlfootball import penalties as pen

pd.set_option("display.width", 110)

df = pen.load_kicks()
print(f"{len(df)} clean kicks across {df.Game_id.nunique()} shootouts; "
      f"overall conversion {df.Goal.mean():.1%}")
df.head()

320 clean kicks across 35 shootouts; overall conversion 69.1%


,Game_id,Team,Zone,Foot,Keeper,OnTarget,Goal,Penalty_Number,Elimination,ShotDir,ShotHt,KeeperRight
0,1,FRA,7,R,R,1.0,1,1,0.0,L,B,False
1,1,GER,9,R,C,1.0,1,2,0.0,R,B,False
2,1,FRA,6,R,L,1.0,1,3,0.0,R,M,False
3,1,GER,2,R,C,1.0,1,4,0.0,C,T,True
4,1,FRA,9,R,L,1.0,1,5,0.0,R,B,False


## 1. The keeper's coin flip

Keepers commit before the ball is struck. Across these shootouts they read the correct
side under half the time — and reading it right only turns a near-certain goal into a
coin flip, it does not make the save automatic.

In [2]:
cf = pen.coin_flip(df)
print(f"keeper reads correct side : {cf['keeper_correct_rate']:.1%}")
print(f"conversion | keeper WRONG  : {cf['conv_when_wrong']:.1%}  (n={cf['n_wrong']})")
print(f"conversion | keeper RIGHT  : {cf['conv_when_right']:.1%}  (n={cf['n_right']})")
print(f"keeper dive distribution   : {cf['keeper_dive_dist']}")

keeper reads correct side : 46.2%
conversion | keeper WRONG  : 84.3%  (n=172)
conversion | keeper RIGHT  : 51.3%  (n=148)
keeper dive distribution   : {'L': 0.4781, 'C': 0.1156, 'R': 0.4062}


## 2. The payoff matrix

The whole game on one grid: rows = where the ball went, columns = where the keeper dived,
each cell = conversion. The diagonal (keeper reads the side) is where saves live; step one
cell off it and the penalty is all but in.

In [3]:
mx = pen.payoff_matrix(df)
print(pd.crosstab(df.ShotDir, df.Keeper, values=df.Goal, aggfunc="mean").round(2))
print("\ncell counts:")
print(pd.crosstab(df.ShotDir, df.Keeper))

Keeper      C     L     R
ShotDir                  
C        0.25  0.79  0.59
L        0.85  0.53  0.88
R        0.94  0.97  0.52

cell counts:
Keeper    C   L   R
ShotDir            
C         8  29  27
L        13  88  51
R        16  36  52


## 3. Is anyone playing Nash?

The clean game-theory prediction: at a mixed-strategy equilibrium every option a taker
uses must pay the same, otherwise they would shift toward the better one. So is conversion
flat across left / centre / right? A high χ² p-value here is a *null worth printing* — it
is exactly what equilibrium play looks like. Unlike the open-data version, this read no
longer leans on the shooter alone: the keeper's real dive is in every cell above.

In [4]:
nash = pen.nash_indifference(df)
for b in nash["by_dir"]:
    print(f"  {b['dir']}: conv {b['conv']:.1%}  used {b['usage']:.1%}  "
          f"95% CI [{b['lo']:.1%}, {b['hi']:.1%}]  (n={b['n']})")
print(f"\n  chi-square goal~side: chi2={nash['chisq']['chi2']}, p={nash['chisq']['p']}, "
      f"spread={nash['spread_pp']} pts")

  L: conv 67.8%  used 47.5%  95% CI [60.0%, 74.7%]  (n=152)
  C: conv 64.1%  used 20.0%  95% CI [51.8%, 74.7%]  (n=64)
  R: conv 74.0%  used 32.5%  95% CI [64.9%, 81.5%]  (n=104)

  chi-square goal~side: chi2=2.074, p=0.3545, spread=10.0 pts


## 4. The abandoned centre

Keepers almost always dive — they hold the middle on barely a tenth of kicks — so a ball
hit straight down the centre usually meets thin air where the keeper just left. The
panenka is a read, not a flourish. (The "keeper stays central" cell is small-n, so treat
its conversion as a warning, not a precise rate.)

In [5]:
c = pen.abandoned_centre(df)
print(f"keeper stays central     : {c['keeper_stays_centre']:.1%}")
print(f"shooter aims centre      : {c['shooter_centre_usage']:.1%}")
print(f"centre vs committed keeper: {c['centre_vs_committed_conv']:.1%} (n={c['centre_vs_committed_n']})")
print(f"centre vs centred keeper  : {c['centre_vs_centre_conv']:.1%} (n={c['centre_vs_centre_n']})")

keeper stays central     : 11.6%
shooter aims centre      : 20.0%
centre vs committed keeper: 69.6% (n=56)
centre vs centred keeper  : 25.0% (n=8)


## 5. Footedness — the one tell left

A right-footer striking across the ball favours the viewer-left; left-footers mirror it.
It is the one place a keeper who knows the taker's stronger foot starts a half-step ahead.

In [6]:
print(pd.DataFrame({f["foot"]: f["dist"] for f in pen.footedness(df)["by_foot"]}).T)

        L       C       R
R  0.5039  0.1914  0.3047
L  0.3594  0.2344  0.4062


## 6. The pressure cooker — honest nulls

Conversion by the taker's order number, and the must-score elimination kicks. If pressure
bites it should show here; it largely doesn't. The confound to keep in mind: kick order is
manager-chosen, so order effects and taker quality are entangled.

In [7]:
pr = pen.pressure(df)
for k in pr["by_kick"]:
    print(f"  kick {k['num']:>2}: {k['conv']:.1%}  (n={k['n']})")
el = pr["elimination"]
print(f"\n  must-score elimination : {el['must_conv']:.1%} (n={el['must_n']})")
print(f"  everything else        : {el['rest_conv']:.1%} (n={el['rest_n']})")

  kick  1: 71.4%  (n=35)
  kick  2: 74.3%  (n=35)
  kick  3: 68.6%  (n=35)
  kick  4: 74.3%  (n=35)
  kick  5: 71.4%  (n=35)
  kick  6: 68.6%  (n=35)
  kick  7: 68.6%  (n=35)
  kick  8: 59.4%  (n=32)
  kick  9: 64.0%  (n=25)
  kick 10: 71.4%  (n=14)

  must-score elimination : 67.4% (n=46)
  everything else        : 69.3% (n=274)


## Emit the bundle

Everything the page needs is one static `site/data/penalties.json` — no runtime compute.

In [8]:
out = pen.export()
print("written site/data/penalties.json:",
      f"{out['meta']['n_clean']} kicks, Nash p={out['nash']['chisq']['p']}, "
      f"keeper correct {out['coin_flip']['keeper_correct_rate']:.1%}")

written site/data/penalties.json: 320 kicks, Nash p=0.3545, keeper correct 46.2%
